In [2]:
%%sql
INSERT INTO gold_zone_hour_metrics_streaming
SELECT
  CAST(date_trunc('day', to_timestamp(tpep_pickup_datetime)) AS date)  AS pickup_date,
  HOUR(to_timestamp(tpep_pickup_datetime))                             AS pickup_hour,
  CAST(PULocationID AS int)                                            AS zone_id,
  COUNT(*)                                                             AS trips_count,
  SUM(total_amount)                                                    AS total_revenue,
  AVG(total_amount)                                                    AS avg_total_amount,
  AVG(trip_distance)                                                   AS avg_trip_distance,
  AVG(
      (UNIX_TIMESTAMP(to_timestamp(tpep_dropoff_datetime)) -
       UNIX_TIMESTAMP(to_timestamp(tpep_pickup_datetime))) / 60.0
  )                                                                    AS avg_trip_duration_min,
  SUM(tip_amount)                                                      AS tip_amount_total,
  CASE WHEN SUM(total_amount) = 0 THEN 0.0
       ELSE SUM(tip_amount) / SUM(total_amount)
  END                                                                  AS tip_share_pct,
  CASE
    WHEN dayofweek(date_trunc('day', to_timestamp(tpep_pickup_datetime))) IN (1,7)
    THEN TRUE ELSE FALSE
  END                                                                  AS is_weekend,
  CURRENT_TIMESTAMP                                                    AS load_timestamp,
  date_trunc('hour', to_timestamp(tpep_pickup_datetime))               AS pickup_ts
FROM silver_zone_hour_metrics_streaming
WHERE
  PULocationID IS NOT NULL
  AND tpep_pickup_datetime IS NOT NULL
  AND tpep_dropoff_datetime IS NOT NULL
  AND total_amount > 0
GROUP BY
  CAST(date_trunc('day', to_timestamp(tpep_pickup_datetime)) AS date),
  HOUR(to_timestamp(tpep_pickup_datetime)),
  CAST(PULocationID AS int),
  date_trunc('hour', to_timestamp(tpep_pickup_datetime));


StatementMeta(, d3691f9d-008d-46d2-b271-1610586a1260, 3, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>

In [3]:
%%sql
INSERT INTO dq_streaming_checks
SELECT
  CURRENT_TIMESTAMP                                             AS check_time,
  (CURRENT_TIMESTAMP - INTERVAL 1 HOUR)                         AS window_start,
  CURRENT_TIMESTAMP                                             AS window_end,
  COUNT(*)                                                      AS total_trips,
  AVG(total_amount)                                             AS avg_total_amount,
  100.0 * SUM(CASE WHEN trip_distance <= 0 THEN 1 ELSE 0 END)
       / COUNT(*)                                               AS bad_distance_pct
FROM silver_zone_hour_metrics_streaming
WHERE
  to_timestamp(tpep_pickup_datetime) >
    CURRENT_TIMESTAMP - INTERVAL 1 HOUR;


StatementMeta(, d3691f9d-008d-46d2-b271-1610586a1260, 4, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>

In [4]:
%%sql
INSERT INTO pipeline_run_log
SELECT
  uuid()                                   AS run_id,
  CURRENT_TIMESTAMP                        AS run_time,
  'silver_zone_hour_metrics_streaming'     AS source_table,
  'gold_zone_hour_metrics_streaming'       AS target_table,
  (SELECT COUNT(*) FROM gold_zone_hour_metrics_streaming) AS rows_written,
  'SUCCESS'                                AS status,
  'Streaming gold refresh completed'       AS message;


StatementMeta(, d3691f9d-008d-46d2-b271-1610586a1260, 5, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>